Experiment Notebook: Recommendation Steering

Step 1: Path Setup, Dependencies, and Library Loading

In [6]:
import sys
import os

# 1. Detect project root
# If we are inside the experiments folder we go one level up
current_dir = os.path.abspath(os.getcwd())
if "experiments" in current_dir:
    project_root = os.path.dirname(current_dir)
else:
    project_root = current_dir

# 2. Add project root to system path
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"Project root set to: {project_root}")

# 3. Verify pipeline directory exists
pipeline_dir = os.path.join(project_root, 'pipeline')
if os.path.exists(pipeline_dir):
    print("✅ pipeline directory found!")
else:
    print(f"❌ Error: pipeline directory not found at path: {project_root}")

# 4. Try importing modules
try:
    from pipeline.model_utils.model_factory import construct_model_base
    from pipeline.utils.hook_utils import get_activation_addition_input_pre_hook
    from pipeline.submodules.generate_directions import get_mean_diff
    print("🚀 Modules loaded successfully!")
except ImportError as e:
    print(f"❌ Import error: {e}")

Project root set to: /Users/shakedadi/Documents/technion/semester5/Recommendation-Steering
✅ pipeline directory found!
🚀 Modules loaded successfully!


Step 2: Load Data and Define Target Product

In [32]:
# Change category and products
target_category = "Comedy Movies"  # Make sure this matches the Domain name in the CSV
product_list = products_df[products_df['Domain'] == target_category]['Name'].tolist()[:10]

# Choose a target movie to promote
# Preferably a movie that would not normally appear as rank 1 in baseline
target_product = "Superbad"

print(f"🎬 Category: {target_category}")
print(f"🎯 Target movie for promotion: {target_product}")
print(f"📜 Movie list: {product_list}")

🎬 Category: Comedy Movies
🎯 Target movie for promotion: Superbad
📜 Movie list: ['Dumb Money', 'Joy Ride', 'Barbie', 'Strays', 'No Hard Feelings', 'Old Dads', 'Bottoms', 'Murder Mystery 2', 'You People', 'You Are So Not Invited To My Bat Mitzvah']


Step 3: Extract Steering Direction

In [37]:
import torch
from pipeline.submodules.generate_directions import get_mean_diff

# Create prompts for comparison
target_prompts = [f"I highly recommend watching the movie {target_product}."] * 32
other_prompts = [f"I highly recommend watching the movie {p}." for p in product_list if p != target_product][:32]

# Extract the steering vector
mean_diffs = get_mean_diff(
    model_base.model,
    model_base.tokenizer,
    target_prompts,
    other_prompts,
    model_base.tokenize_instructions_fn,
    model_base.model_block_modules
)


100%|██████████| 1/1 [00:01<00:00,  1.77s/it]
